The IW-IB-5GNET is a dataset that can be used to carry out research for developing AI-powered autonomous 5G and beyond networks. It focuses on the Infrastructure-wide and Intent-Based networking aspects to offer definite information about the 5G network behavior, management and, automation processes. This extract has 1000 examples, so it can be used in practical experiments and for the purpose of researching the applicability of the methods described here, while the full dataset contains about 36000 examples.

 Such information includes bandwidth utilization, delay, resource provisioning, traffic flow, and topological/or device-specific data at various nodes of a network. It also allows studying how networks would behave under certain operational conditions such as the intent-based network management that is geared towards auto-provisioning of networking resources based on business intents.

 This dataset is especially suitable for our case of modeling VNF deployment and anticipating performance issues. It provides infrastructure-wide metrics which can be used to train other machine learning algorithms such as Random forest and LSTM to predict the network and plan resource usage as well as look into areas of potential inefficiency and redress them. As for its compatibility for or use case, it is quite appropriate to choose this one because the AI-driven aspect complements our goal of using ML algorithms in enhancing the network performance and relatively autonomous management.

In [ ]:
# Import libraries for data analysis and visualization
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# To display all columns
pd.set_option('display.max_columns', None)

# For dealing with missing values
import missingno as msno


In [ ]:
# Load the dataset
df = pd.read_csv('/content/iwib5gnet_v0.csv')

In [ ]:
# Select relevant columns
relevant_columns = ['OVS_RX_bytes', 'OVS_TX_bytes', 'TC_RX_bytes', 'TC_TX_bytes',
                    'totalpktCount', 'totalBits', 'packetSize', 'ruleComplexity', 'timestamp']

In [ ]:
df = df[relevant_columns]

# Drop rows with missing values
df = df.dropna()

In [ ]:
# Convert timestamp to datetime and extract features
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
df['timestamp'] = df['timestamp'].astype(int) / 10**9  # Convert to seconds


In [ ]:
# Keep only relevant columns and drop the rest
df_clean = df[relevant_columns]

# Check for missing values
df_clean = df_clean.dropna()

In [ ]:
df.info()

In [ ]:
# Convert timestamp to datetime for time-series analysis (optional)
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'], unit='s')

In [ ]:
# Normalize the numerical features for Random Forest and LSTM
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerical_columns = ['ContextSwitchesPerSecond', 'iface_speed', 'OVS_RX_bytes',
                     'OVS_TX_bytes', 'TC_RX_bytes', 'TC_TX_bytes',
                     'totalpktCount', 'totalBits', 'packetSize',
                     'averageMatchedBytes', 'averageMatchedPackets', 'ruleComplexity']

In [ ]:
df_clean[numerical_columns] = scaler.fit_transform(df_clean[numerical_columns])

# Print first few rows to check the processed data
print(df_clean.head())

In [ ]:
# Descriptive statistics to understand the data distribution
print(df.describe())

In [ ]:
# Check for missing values
print(df.isnull().sum())

In [ ]:
# 2. Histogram: Packet Sizes Distribution
plt.figure(figsize=(10, 6))
sns.histplot(df['packetSize'], bins=30, kde=True, color='purple')
plt.title('Distribution of Packet Sizes')
plt.xlabel('Packet Size (bytes)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# 3. Bar Plot: Rule Complexity vs. Total Packets
plt.figure(figsize=(10, 6))
sns.barplot(x='ruleComplexity', y='totalpktCount', data=df, palette='coolwarm')
plt.title('Rule Complexity vs. Total Packet Count')
plt.xlabel('Rule Complexity')
plt.ylabel('Total Packets')
plt.show()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

In [ ]:
# Check and convert all columns to numeric if necessary
df = df.apply(pd.to_numeric, errors='ignore')


In [ ]:
# Extract features and target
features = df.drop(['timestamp'], axis=1)
target = features.pop('totalpktCount')

# Normalize features
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(features_scaled, target, test_size=0.2, random_state=42)

##Random Forest Model

In [ ]:
# Train Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predictions and evaluation
rf_predictions = rf_model.predict(X_test)
rf_mse = mean_squared_error(y_test, rf_predictions)
print(f'Random Forest Mean Squared Error: {rf_mse}')


###LSTM Model Preparation

In [ ]:
# Convert data to sequences for LSTM
def create_sequences(data, target, seq_length):
    sequences = []
    targets = []
    for i in range(len(data) - seq_length):
        sequences.append(data[i:i+seq_length])
        targets.append(target[i+seq_length])
    return np.array(sequences), np.array(targets)

seq_length = 10
X_seq, y_seq = create_sequences(features_scaled, target.values, seq_length)

# Train-test split for LSTM
X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(X_seq, y_seq, test_size=0.2, random_state=42)


In [ ]:
# Define LSTM model
lstm_model = Sequential()
lstm_model.add(LSTM(50, activation='relu', input_shape=(X_train_seq.shape[1], X_train_seq.shape[2])))
lstm_model.add(Dense(1))
lstm_model.compile(optimizer='adam', loss='mse')


In [ ]:
# Train LSTM model
lstm_model.fit(X_train_seq, y_train_seq, epochs=10, batch_size=32, validation_split=0.1, verbose=1)

# Predictions and evaluation
lstm_predictions = lstm_model.predict(X_test_seq)
lstm_mse = mean_squared_error(y_test_seq, lstm_predictions)
print(f'LSTM Mean Squared Error: {lstm_mse}')

In [ ]:
from tensorflow.keras.optimizers import Adam

# Experiment with different configurations
lstm_model = Sequential()
lstm_model.add(LSTM(100, activation='relu', input_shape=(X_train_seq.shape[1], X_train_seq.shape[2])))
lstm_model.add(Dense(1))
lstm_model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

lstm_model.fit(X_train_seq, y_train_seq, epochs=20, batch_size=32, validation_split=0.1, verbose=1)

# Re-evaluate
lstm_predictions = lstm_model.predict(X_test_seq)
lstm_mse = mean_squared_error(y_test_seq, lstm_predictions)
print(f'Tuned LSTM Mean Squared Error: {lstm_mse}')


#Explanation
The implementation involves data analysis where data regarding 5G network traffic is analysed with a view of identifying VNF performance bottlenecks and improving the performance of these VNFs using machine learning models. Specific measurements of the networks, including the packets and bytes transmitted, and parameters corresponding to the management components are Open vSwitch (OVS), Traffic Control (TC) and IP Tables (IPTAB), are stored and labeled as, IW-IB-5GNET.

 Data Preparation and Processing

 Data Preparation: The first process included the process of feature selection where only the most relevant features from the data set were used during the training of the model. As dependent variables, we included numeric columns related to the network traffic volume and selected in disjunction with categorical and non-numeric fields. This step of text preprocessing contributes to a good result in machine learning models because these models need numeric data. The time field was also converted to seconds for numerical uniformity while missing values in the dataset were dealt with by row elimination.

 Feature Scaling: Log transformation was then applied following which the features were normalized for scale using the StandardScaler. This step is very crucial especially for the models such as LSTM, which are very much affected by the scale of the input data. Normalization guarantees that all features have the same importance in the model’s performance.

 Machine Learning Models

 Random Forest Model: Thus, a Random Forest Regressor was used for network traffic metrics’ prediction based on the preprocessed features. Random Forest is a type of methods to amalgamate the multiple decision trees with the purpose of increasing the rate of accuracy of prediction besides the over fitting of data. Looking at its performance, that had a Mean Squared Error (MSE) of 158,408. 35 which makes it a good predictor with the presented data set. Because Random Forest is rather insensitive to the type of data and is really easy to implement, we should consider it a good candidate for the first models in the process.


LSTM Model: For capturing sequential dependencies in the data, the Long Short-Term Memory (LSTM) network, a category of Recurrent Neural Network (RNN) was used. LSTMs happen to be ideal for time-series prediction since the model learns from sequences of time. However, the LSTM model had a comparatively higher MSE, that is 19,851,024. On average, it took AL 10 to solve the current dataset configuration problem stating that it was difficult for it to figure out how to process the configurations effectively.

 Challenges and Improvements: There are a number of concerns which can be raised based on the poor performance of Random Forest model with respect to LSTM:

 Sequence Preparation: It should be appreciated that sequence generation for LSTM is critical. If sequence length is misplaced or wrong the performance of the model will be affected.
 Hyperparameter Tuning: Compared with traditional RNNs, LSTMs involve considerable processing based on such hyperparameters as quantity of layers, units, and learning rate. The current formalization of the LSTM could be influenced and tuned on these parameters to increase the accuracy of the prediction.
 Feature Engineering: It can be seen that improving feature selection and engineering can offer LSTM models with better inputs which might lead into an improvement in their performance.

 Conclusion
 Concisely, this project is about applying machine learning to the reconfigurable network functions to get better performances in a 5G environment. It is apparent that the Random Forest model gives a good starting point and is good enough for the acceptable level of performance but the LSTM model has slightly higher MSE principally requiring further optimization. The approach entails data pre- processing, feature creation, and model building to solve network management problems toward improving the performance of the 5G networks.